# Topic 01 — Exploratory Data Analysis with Pandas

Практическая работа по датасету **UCI Adult**. Ноутбук основан на открытом demo assignment курса mlcourse.ai (Yury Kashnitsky, CC BY-NC-SA 4.0), но решение, структура и пояснения оформлены самостоятельно.

Цель: закрепить загрузку и исследование данных, фильтрацию, `loc`, булевы маски, `value_counts`, `groupby`, `agg`, описательные статистики и формулирование выводов.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/adult.data.csv'
LOCAL_DATA = Path('../data/adult.data.csv')

if LOCAL_DATA.exists():
    data = pd.read_csv(LOCAL_DATA)
else:
    data = pd.read_csv(DATA_URL)

data.head()

## 1. Первичный осмотр

Перед ответами на вопросы проверяем размер, типы данных, пропуски и базовые статистики. Это обязательный этап: нельзя анализировать таблицу, не понимая её структуру.


In [ ]:
print('Shape:', data.shape)
display(data.head())
data.info()
display(data.describe(include='all').T)

## 2. Сколько мужчин и женщин представлено в данных?


In [ ]:
sex_counts = data['sex'].value_counts()
sex_counts

**Ожидаемый результат:** 21 790 мужчин и 10 771 женщина. `value_counts()` — базовый инструмент для частот категориального признака.


## 3. Средний возраст женщин


In [ ]:
female_mean_age = data.loc[data['sex'] == 'Female', 'age'].mean()
female_mean_age

**Ответ:** примерно **36.86 года**. Здесь `.loc[условие, столбец]` сразу выражает и фильтрацию строк, и выбор нужного признака.


## 4. Доля граждан Германии


In [ ]:
germany_share = data['native-country'].eq('Germany').mean()
germany_share, germany_share * 100

**Ответ:** около **0.42%**. Булевы значения в Pandas интерпретируются как `1/0`, поэтому среднее булевой маски равно доле `True`.


## 5. Возраст и уровень дохода


In [ ]:
age_by_salary = data.groupby('salary')['age'].agg(['mean', 'std'])
age_by_salary

**Вывод:** люди с доходом `>50K` в среднем старше: около **44.25 года** против **36.78 года** у группы `<=50K`. Стандартные отклонения — примерно **10.52** и **14.02** года соответственно. Это описательная связь, а не доказательство причинности.


## 6. Все ли люди с доходом >50K имеют высшее образование?


In [ ]:
higher_education = {'Bachelors', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', 'Masters', 'Doctorate'}
high_income_education = data.loc[data['salary'] == '>50K', 'education']
unexpected = sorted(set(high_income_education) - higher_education)
unexpected

**Ответ:** нет. В группе `>50K` встречаются люди без перечисленных степеней высшего образования, например `HS-grad` и `Some-college`. Проверяем утверждение данными, а не предположением.


## 7. Возраст по расе и полу


In [ ]:
age_stats = data.groupby(['race', 'sex'])['age'].describe()
display(age_stats)
amer_indian_male_max = age_stats.loc[('Amer-Indian-Eskimo', 'Male'), 'max']
amer_indian_male_max

**Ответ:** максимальный возраст мужчин `Amer-Indian-Eskimo` — **82 года**. Многоуровневая группировка позволяет получать статистику по комбинациям категорий.


## 8. Доход >50K среди женатых и неженатых мужчин


In [ ]:
men = data[data['sex'] == 'Male'].copy()
men['married'] = men['marital-status'].str.startswith('Married')
high_income_share = men.assign(high_income=men['salary'].eq('>50K')).groupby('married')['high_income'].mean().mul(100)
high_income_share.rename(index={False: 'Not married', True: 'Married'})

**Вывод:** доля высокодоходных заметно выше среди женатых мужчин. Важно сравнивать именно **доли внутри групп**, а не абсолютное число людей.


## 9. Максимальная рабочая неделя


In [ ]:
max_hours = data['hours-per-week'].max()
max_workers = data[data['hours-per-week'] == max_hours]
n_max_workers = len(max_workers)
high_income_pct = max_workers['salary'].eq('>50K').mean() * 100
max_hours, n_max_workers, high_income_pct

**Ответ:** максимум — **99 часов в неделю**, так работают **85 человек**, из них около **29.4%** получают `>50K`. Большое число часов само по себе не гарантирует высокий доход.


## 10. Среднее рабочее время по стране и доходу


In [ ]:
hours_by_country_salary = (
    data.groupby(['native-country', 'salary'])['hours-per-week']
        .mean()
        .unstack('salary')
        .sort_index()
)
hours_by_country_salary

## Итоги Topic 01

В этой работе использованы ключевые операции Pandas: `read_csv`, `head`, `info`, `describe`, `value_counts`, булевы маски, `.loc`, строковые методы, `groupby`, `agg`, `unstack`, `mean`, `std`, `max`.

Главный навык — не знание списка методов, а перевод аналитического вопроса в последовательность операций: **выбрать наблюдения → выбрать признаки → агрегировать → проверить → интерпретировать**.
